# AuroraStay Policy RAG — completed low-code notebook
Purpose: answer internal staff questions from four supplied policy documents, show page-level sources, and measure all 20 supplied cases. The PDF presentation is the required submission; this notebook is a supporting, reproducible artifact.

**Execution note:** this notebook runs the local FAISS/TF-IDF baseline without secrets. Its quality metrics are offline lexical proxies, not DeepEval judge scores. Hosted LLM evaluation and Codespaces inference remain separate deployment steps.

In [1]:
from pathlib import Path
import json, sys
import pandas as pd
ROOT = Path.cwd()
if not (ROOT / 'Data').exists():
    ROOT = Path('aurorastay_rag_poc')
sys.path.insert(0, str(ROOT / 'backend'))
from rag_core import TfidfFaissStore, read_policy_pages, chunk_pages, public_result
print('Project:', ROOT.resolve())

Project: /home/runner/workspace/aurorastay_rag_poc


## 1. Data understanding and document extraction

In [2]:
pages = read_policy_pages(ROOT / 'Data')
chunks = chunk_pages(pages, chunk_words=120, overlap=20)
print(f'Policy PDFs: {len(list((ROOT / "Data").glob("*.pdf")))}')
print(f'Extracted pages: {len(pages)} | chunks: {len(chunks)}')
print('Sample metadata:', {k: chunks[0][k] for k in ('source','page')})
print('Sample chunk:', chunks[0]['text'][:400])

Policy PDFs: 4
Extracted pages: 19 | chunks: 61
Sample metadata: {'source': 'aurorastay_booking_terms_and_cancellation_policy.pdf', 'page': 1}
Sample chunk: AuroraStay Hotels | AS-01 | Internal operating policy Page 1 AURORASTAY HOTELS Booking Terms and Cancellation Policy Individual transient reservations | US properties | Guest-facing baseline with operational controls Policy code AS-01 Effective date January 1, 2026 Review date January 1, 2027, or sooner if law, systems, or operating conditions change Owner Guest Experience and Property Operations 


## 2. Chunking, local embeddings, and vector database
Each page is split into 120-word chunks with 20-word overlap. The local corpus-fitted TF-IDF embedder creates normalized vectors; FAISS IndexFlatIP returns cosine-ranked passages. This avoids model downloads and retains source filename/page metadata.

In [3]:
index_dir = ROOT / 'backend' / 'index'
if (index_dir / 'store.json').exists():
    store = TfidfFaissStore.load(index_dir)
else:
    store = TfidfFaissStore.build(chunks)
    store.save(index_dir)
print('FAISS vectors:', store.index.ntotal)
print('Vector dimensions:', len(store.vocabulary))

FAISS vectors: 61
Vector dimensions: 1008


## 3. Retrieval and grounded answer demonstration

In [4]:
sample_question = 'What is the pet fee at ASH-CHI Loop?'
sample = public_result(sample_question, k=3, store=store)
print('Question:', sample_question)
print('Answer:', sample['answer'])
print('Mode:', sample['mode'])
for item in sample['relevant_chunks']:
    print(f"\n{item['source']} p.{item['page']} (score={item['score']})")
    print(item['content'][:350])

Question: What is the pet fee at ASH-CHI Loop?
Answer: Property Parking Pet fee Breakfast ASH-ATL Midtown $28 self parking per night $75 per stay $19 adult, $10 child ASH-BOS Harbor $65 valet per night No pets except service animals Included ASH-CHI Loop $42 self parking per night $50 per night, $150 stay cap $22 per person ASH-DEN Union $35 garage per night $75 per stay $18 per person ASH-MIA Brickell $55 valet per night $100 per stay $24 per person ASH-SEA Pike $38 partner garage per night $75 per stay Included 6. [aurorastay_refunds_and_additional_charges_guide.pdf, p. 4]
Mode: offline_extract

aurorastay_refunds_and_additional_charges_guide.pdf p.4 (score=0.274)
AuroraStay Hotels | AS-03 | Internal operating policy Page 4 5. Property-specific charge examples Use this table only as a property lookup example. Verify the live profile and the guest's rate confirmation before quoting a charge. Property Parking Pet fee Breakfast ASH-ATL Midtown $28 self parking per night $75 per stay $19

## 4. Evaluation and held-out test set
Run all ten rows from each provided file. The four local checks below are labelled proxies: retrieved-context term recall/precision, exact extractive support, and query-term answer overlap. They are useful for debugging but are not semantic judge metrics. For official GEval scores, run DeepEval with an approved judge-model secret and report its actual per-case results.

In [5]:
import subprocess
subprocess.run([sys.executable, str(ROOT / 'backend' / 'evaluate_offline.py')], check=True)
results = pd.read_csv(ROOT / 'evaluation_results_offline.csv')
summary = json.loads((ROOT / 'evaluation_summary_offline.json').read_text())
display(pd.DataFrame(summary).T)
display(results[['split','case','question','sources','context_recall_proxy','context_precision_proxy','extractive_support_proxy','answer_relevancy_proxy']])

 cases  context_recall_proxy  context_precision_proxy  extractive_support_proxy  answer_relevancy_proxy  answers_returned
  10.0                 0.574                    0.164                       1.0                   0.449              10.0
  10.0                 0.704                    0.158                       1.0                   0.455              10.0
     split  case                                                                                                                        question                                                                                                                                                                                     sources  context_recall_proxy  context_precision_proxy  extractive_support_proxy  answer_relevancy_proxy
Evaluation     1                                                         What time can I check in at ASH-CHI Loop, and when do I need to be out?        aurorastay_guest_conduct_accessibility_and_resolutio

## 5. Deployment and completion status
- Backend files: `backend/app.py`, `backend/rag_core.py`, `backend/requirements.txt`, `backend/Dockerfile`.
- Frontend files: `frontend/app.py`, `frontend/requirements.txt`, `frontend/Dockerfile`; `docker-compose.yml` wires them together.
- Routes: `/health`, `/v1/relevant_chunks`, `/v1/answer`, `/v1/answer_with_relevant_chunks`.
- Local index/inference can be tested. GitHub push and public Codespaces forwarded-URL inference have not been performed; no deployment URL or screenshot is fabricated.
- Keep API credentials only in the hosting provider's secret manager.